# 생성형 인공지능 · 3주차 실습
## Diffusion 5장 — 특수한 구조의 데이터에 확산 모델링 적용하기

**대상**: 전공 4학년 · **환경**: Google Colab (CPU 런타임으로 충분, 추가 설치 불필요)
**소요 시간**: 약 2~3시간

---

### 실습 개요

강의 슬라이드는 "일반적인 가우시안 확산을 그대로 쓸 수 없는 데이터"를 다뤘습니다.
이 노트북은 그 네 가지 상황을 각각 코드로 구현해 봅니다.

| # | 주제 | 슬라이드 대응 | 핵심 개념 |
|---|------|--------------|-----------|
| Q1 | 이산 확산의 전이 커널 | 불연속형 데이터 · VQ-Diffusion | $q(x_t\|x_{t-1}) = v^\top(x_t) Q_t v(x_{t-1})$ |
| Q2 | 마스크 흡수 커널과 VLB 손실 | VQ-Diffusion `_train_loss` | 사후분포, multinomial KL, 마스크 가중치 |
| Q3 | 그래프 확산 (GDSS) | 불변 구조의 데이터 · GDSS | VP-SDE 섭동, DSM 손실, 순열 등변성 |
| Q4 | SE(3) 등변 메시지 전달 | GeoDiff · SchNetEncoder / CFConv | 회전·병진 불변 피처, 등변 좌표 업데이트 |
| Q5 | 리만 다양체 위의 확산 | 스트리밍(매니폴드) 구조 데이터 | 접공간 투영, 지수 사상, 측지 확산 |
| Q6 | 잠재 확산 모델 (LDM) | 알 수 없는 흐름 패턴 · Stable Diffusion | `DiagonalGaussianDistribution`, 잠재공간 DDPM |
| Q7 | 크로스 어텐션 조건부 생성 | Stable Diffusion `CrossAttention` | 텍스트 조건화, 마스킹, CFG |

### 진행 방법

1. 각 문제의 `# TODO` 부분을 채워 넣습니다.
2. 바로 아래 **검증 셀**을 실행합니다. `assert` 가 모두 통과하면 `✅ Qn 통과` 가 출력됩니다.
3. 마지막 서술형 질문에는 마크다운 셀에 3~5줄로 답합니다.

### 채점 배점 (100점)

| 항목 | 배점 |
|------|------|
| Q1 이산 전이 커널 | 12 |
| Q2 마스크 흡수 + KL 손실 | 16 |
| Q3 그래프 확산 + 등변성 | 16 |
| Q4 SE(3) 등변성 | 16 |
| Q5 리만 확산 | 12 |
| Q6 잠재 확산 | 16 |
| Q7 크로스 어텐션 | 12 |

## 0. 공통 설정

아래 셀을 가장 먼저 실행하세요. Colab 기본 이미지에 PyTorch 가 이미 설치되어 있어 추가 설치는 필요 없습니다.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(s=0):
    np.random.seed(s)
    torch.manual_seed(s)

set_seed(0)
device = torch.device("cpu")      # 본 실습은 전 문제 CPU 로 수 분 내에 끝납니다.
ATOL, RTOL = 1e-5, 1e-4           # 검증용 허용 오차

print("torch:", torch.__version__, "| device:", device)

---
# Q1. 이산 확산 모델의 전이 커널 (12점)

## 배경

대부분의 확산 모델은 **연속형** 데이터를 다룹니다. 하지만 VQ-VAE 로 이미지를 토큰으로 이산화하면
가우시안 노이즈를 더할 수 없으므로, 순방향 과정을 **범주형 전이 행렬**로 대체합니다 (VQ-Diffusion).

슬라이드의 전이 커널은 다음과 같습니다.

$$q(x_t \mid x_{t-1}) = v^\top(x_t)\, Q_t\, v(x_{t-1}), \qquad
Q_t = \begin{bmatrix}\alpha_t+\beta_t & \beta_t & \cdots & \beta_t\\
\beta_t & \alpha_t+\beta_t & \cdots & \beta_t\\
\vdots & & \ddots & \vdots\\
\beta_t & \beta_t & \cdots & \alpha_t+\beta_t\end{bmatrix}$$

여기서 $v(\cdot)$ 는 원-핫 벡터, $Q_t[i,j] = q(x_t=i \mid x_{t-1}=j)$ 이며 $\alpha_t + K\beta_t = 1$ 입니다.
즉 **확률 $\alpha_t$ 로 토큰을 유지하고, 확률 $K\beta_t$ 로 $K$개 토큰 중 하나를 균등하게 다시 뽑습니다.**

## 과제

1. `make_Qt(beta_t, K)` : 위 전이 행렬을 만드세요.
2. `cumulative_Q(betas, K)` : 누적 전이 행렬 $\bar{Q}_t = Q_t Q_{t-1}\cdots Q_1$ 을 $t=0,\dots,T$ 에 대해 반환하세요 ($\bar{Q}_0 = I$).
3. `cumulative_Q_closed_form(betas, K)` : 행렬 곱 없이 닫힌 형태
   $\bar{Q}_t = \bar\alpha_t I + \frac{1-\bar\alpha_t}{K}\mathbf{1}\mathbf{1}^\top,\quad \bar\alpha_t=\prod_{s\le t}\alpha_s$
   로 계산하세요. (2번과 같은 값이 나와야 합니다.)
4. `sample_forward(x0, betas, K)` : 실제로 한 스텝씩 토큰을 샘플링해 $x_0,\dots,x_T$ 를 반환하세요.

**힌트**: $J = \mathbf{1}\mathbf{1}^\top$ 에 대해 $J^2 = KJ$ 임을 쓰면 3번 닫힌 형태를 유도할 수 있습니다.
샘플링은 `torch.multinomial` 을 쓰고, `Qt.T[x]` 가 "이전 상태 x 에서 출발하는 전이 확률 행"이 됩니다.

In [ ]:
def make_Qt(beta_t, K):
    """단일 스텝 전이 행렬 Q_t. Q[i, j] = q(x_t = i | x_{t-1} = j) (열-확률 행렬)."""
    beta_t = torch.as_tensor(beta_t, dtype=torch.float32)
    # TODO: alpha_t = 1 - K * beta_t 를 이용해 (K, K) 전이 행렬을 만드세요.
    raise NotImplementedError


def cumulative_Q(betas, K):
    """Qbar[t] = Q_t Q_{t-1} ... Q_1 을 t=0..T 에 대해 쌓아서 (T+1, K, K) 로 반환. Qbar[0] = I."""
    # TODO: 리스트에 항등행렬부터 넣고, 매 스텝 make_Qt(...) 를 왼쪽에서 곱해 나가세요.
    raise NotImplementedError


def cumulative_Q_closed_form(betas, K):
    """Qbar_t = abar_t * I + (1 - abar_t)/K * J  (행렬 곱 없이)."""
    betas = torch.as_tensor(betas, dtype=torch.float32)
    # TODO: alphas = 1 - K*betas, abar = 누적곱(앞에 1 을 붙일 것) 을 이용하세요.
    raise NotImplementedError


def sample_forward(x0, betas, K):
    """x0: (B,) 정수 토큰. 한 스텝씩 샘플링하여 (T+1, B) 정수 텐서를 반환."""
    # TODO: 매 스텝 Qt.T[x] 를 확률로 하여 torch.multinomial 로 다음 토큰을 뽑으세요.
    raise NotImplementedError

In [ ]:
# ---------------- Q1 검증 ----------------
set_seed(0)
K, T = 5, 100
betas = torch.linspace(1e-3, 0.9 / K, T)          # 항상 alpha_t = 1 - K*beta_t >= 0

Qt = make_Qt(betas[10], K)
assert Qt.shape == (K, K)
assert torch.allclose(Qt.sum(0), torch.ones(K), atol=ATOL), "각 열의 합이 1 이어야 합니다(확률 분포)."
assert torch.allclose(Qt, Qt.T, atol=ATOL), "균등 커널이므로 대칭이어야 합니다."

Qbar_prod = cumulative_Q(betas, K)
Qbar_form = cumulative_Q_closed_form(betas, K)
assert Qbar_prod.shape == (T + 1, K, K)
assert torch.allclose(Qbar_prod[0], torch.eye(K), atol=ATOL), "Qbar_0 = I"
assert torch.allclose(Qbar_prod, Qbar_form, atol=1e-4), "행렬 곱과 닫힌 형태가 일치해야 합니다."
assert torch.allclose(Qbar_prod.sum(1), torch.ones(T + 1, K), atol=1e-4)

# t -> T 에서 균등분포로 수렴
uniform = torch.full((K, K), 1.0 / K)
assert torch.allclose(Qbar_prod[-1], uniform, atol=1e-3), "충분히 큰 t 에서 균등분포로 수렴해야 합니다."

# 경험적 주변분포 == 이론적 Qbar_t 의 x0 열
set_seed(1)
B, t_chk, x0_val = 20000, 20, 2
x0 = torch.full((B,), x0_val, dtype=torch.long)
traj = sample_forward(x0, betas, K)
assert traj.shape == (T + 1, B)
emp = torch.bincount(traj[t_chk], minlength=K).float() / B
theo = Qbar_prod[t_chk][:, x0_val]
print("경험적 분포:", np.round(emp.numpy(), 4))
print("이론적 분포:", np.round(theo.numpy(), 4))
assert (emp - theo).abs().max() < 0.02, "샘플링 주변분포가 Qbar_t 와 달라요."

print("✅ Q1 통과")

---
# Q2. 마스크 흡수(mask-absorbing) 커널과 VLB 손실 (16점)

## 배경

슬라이드의 VQ-Diffusion 학습 코드(`_train_loss`)는 균등 재배치뿐 아니라 **`[MASK]` 토큰**을 사용합니다.
상태 공간을 $K$개 실제 토큰 + 1개 `[MASK]`(인덱스 $K$)로 두고,

$$q(x_t\mid x_{t-1}) : \quad
\text{확률 } \alpha_t \text{ 로 유지}, \quad
\text{확률 } K\beta_t \text{ 로 균등 재배치}, \quad
\text{확률 } \gamma_t \text{ 로 } [\text{MASK}], \qquad \alpha_t + K\beta_t + \gamma_t = 1$$

`[MASK]` 는 **흡수 상태(absorbing state)** 라서 한 번 마스킹되면 절대 빠져나오지 못합니다.

학습 손실의 핵심 항은
$$L_{t-1} = \mathrm{KL}\big(q(x_{t-1}\mid x_t, x_0)\;\|\;p_\theta(x_{t-1}\mid x_t)\big)$$
이고, 슬라이드 코드는 여기에 **마스크 영역 가중치**(`mask_weight`)를 곱합니다.

## 과제

1. `make_Qt_mask(alpha_t, beta_t, gamma_t, K)` : $(K{+}1)\times(K{+}1)$ 전이 행렬. 마지막 상태는 흡수 상태.
2. `q_posterior(Qt, Qbar_prev, xt, x0, K)` :
   $$q(x_{t-1}\mid x_t,x_0) \;\propto\; \big[Q_t^\top v(x_t)\big] \odot \big[\bar{Q}_{t-1} v(x_0)\big]$$
   를 정규화해 반환하세요. (`Qt.T @ v(x_t)` 는 결국 **$Q_t$ 의 $x_t$ 번째 행**입니다.)
3. `multinomial_kl(log_p, log_q)` : 슬라이드 코드의 `multinomial_kl` 과 동일하게 로그 확률 두 개를 받아 KL 을 계산.
4. `vb_term(log_q_post, log_p_model, xt, K, mask_weight)` : KL 에 마스크 가중치를 적용한 배치 평균 손실.
   슬라이드의 `mask_region = (xt == num_classes-1)` 부분을 그대로 구현하면 됩니다.

**힌트**: 로그 영역 계산이 안정적이므로 `torch.log(p + 1e-30)` 대신 `p.clamp_min(1e-30).log()` 를 쓰세요.

In [ ]:
def make_Qt_mask(alpha_t, beta_t, gamma_t, K):
    """(K+1, K+1) 전이 행렬. 인덱스 K 는 [MASK] 이며 흡수 상태."""
    assert abs(float(alpha_t) + K * float(beta_t) + float(gamma_t) - 1.0) < 1e-5, "alpha + K*beta + gamma = 1 이어야 합니다."
    Q = torch.zeros(K + 1, K + 1)
    # TODO: (1) 실제 토큰끼리는 유지(alpha) + 균등 재배치(beta)
    #       (2) 실제 토큰 -> [MASK] 는 gamma
    #       (3) [MASK] -> [MASK] 는 확률 1 (흡수)
    raise NotImplementedError


def q_posterior(Qt, Qbar_prev, xt, x0, K):
    """q(x_{t-1} | x_t, x_0) 를 (B, K+1) 확률로 반환."""
    # TODO: left = Q_t 의 x_t 번째 "행", right = Qbar_{t-1} 의 x_0 번째 "열"
    #       두 벡터를 곱한 뒤 정규화하세요.
    raise NotImplementedError


def multinomial_kl(log_prob1, log_prob2):
    """KL(p1 || p2). 입력은 마지막 축이 범주인 로그 확률."""
    # TODO: sum p1 * (log p1 - log p2)
    raise NotImplementedError


def vb_term(log_q_post, log_p_model, xt, K, mask_weight=(5.0, 1.0)):
    """마스크 영역 가중치를 적용한 KL 손실의 배치 평균."""
    # TODO: mask_region = (xt == K) 를 이용해 가중치를 만들고 KL 에 곱한 뒤 평균내세요.
    raise NotImplementedError

In [ ]:
# ---------------- Q2 검증 ----------------
set_seed(0)
K, T = 4, 60
gammas = torch.linspace(0.005, 0.12, T)               # 마스킹 확률
betas_m = torch.linspace(1e-3, 0.02, T)               # 균등 재배치 확률
alphas_m = 1.0 - K * betas_m - gammas

# (1) 전이 행렬 성질
Qm = make_Qt_mask(alphas_m[5], betas_m[5], gammas[5], K)
assert Qm.shape == (K + 1, K + 1)
assert torch.allclose(Qm.sum(0), torch.ones(K + 1), atol=ATOL), "각 열의 합 = 1"
assert Qm[K, K] == 1.0 and torch.allclose(Qm[:K, K], torch.zeros(K)), "[MASK] 는 흡수 상태여야 합니다."

# (2) 누적 커널: 마스킹 확률은 단조 증가하고 1 로 수렴
Qbars_m = [torch.eye(K + 1)]
for t in range(T):
    Qbars_m.append(make_Qt_mask(alphas_m[t], betas_m[t], gammas[t], K) @ Qbars_m[-1])
Qbars_m = torch.stack(Qbars_m)
p_mask = Qbars_m[:, K, 0]                              # x0 = 0 에서 출발했을 때 [MASK] 확률
assert torch.all(p_mask[1:] >= p_mask[:-1] - 1e-6), "마스킹 확률은 단조 증가해야 합니다."
assert p_mask[-1] > 0.95, f"마지막 스텝 마스킹 확률이 너무 낮습니다: {p_mask[-1]:.3f}"

# (3) 순수 흡수 커널(beta=0)의 사후분포는 x_t 가 실제 토큰이면 델타
Qt_abs = make_Qt_mask(0.9, 0.0, 0.1, K)
Qbar_prev_abs = make_Qt_mask(0.8, 0.0, 0.2, K)
x0 = torch.tensor([1, 1, 3])
xt = torch.tensor([1, K, 3])                           # 두 번째 샘플만 마스킹된 상태
post = q_posterior(Qt_abs, Qbar_prev_abs, xt, x0, K)
assert post.shape == (3, K + 1)
assert torch.allclose(post.sum(-1), torch.ones(3), atol=ATOL), "사후분포 합 = 1"
assert post[0, 1] > 1 - 1e-5, "x_t 가 이미 언마스크된 토큰이면 x_{t-1} 도 같은 토큰이어야 합니다."
assert post[2, 3] > 1 - 1e-5
assert post[1, K] > 0 and post[1, x0[1]] > 0, "마스크 상태의 사후분포는 {x_0, [MASK]} 위에 있습니다."
assert post[1, 0] < 1e-6 and post[1, 2] < 1e-6, "흡수 커널에서는 다른 토큰이 나올 수 없습니다."

# (4) KL 성질
lp = torch.log_softmax(torch.randn(8, K + 1), dim=-1)
lq = torch.log_softmax(torch.randn(8, K + 1), dim=-1)
assert torch.allclose(multinomial_kl(lp, lp), torch.zeros(8), atol=1e-6), "KL(p||p) = 0"
assert torch.all(multinomial_kl(lp, lq) > 0), "KL >= 0"

# (5) 마스크 가중치
log_q = post.clamp_min(1e-30).log()
loss_perfect = vb_term(log_q, log_q, xt, K, mask_weight=(5.0, 1.0))
assert loss_perfect.abs() < 1e-5, "완벽한 모델의 KL 손실은 0 이어야 합니다."

log_p_model = torch.log_softmax(torch.randn(3, K + 1), dim=-1)
kl_raw = multinomial_kl(log_q, log_p_model)
expected = (kl_raw * torch.tensor([1.0, 5.0, 1.0])).mean()
got = vb_term(log_q, log_p_model, xt, K, mask_weight=(5.0, 1.0))
assert torch.allclose(got, expected, atol=1e-5), "마스크 영역에만 mask_weight[0] 이 적용되어야 합니다."
print(f"마지막 스텝 마스킹 확률 = {p_mask[-1]:.4f}, 가중 KL = {got.item():.4f}")

print("✅ Q2 통과")

---
# Q3. 그래프 확산 모델 GDSS: 섭동 · DSM 손실 · 순열 등변성 (16점)

## 배경

그래프 $G = (X, A)$ 는 **노드 번호를 바꿔도 같은 그래프**입니다(순열 불변).
GDSS 는 노드 피처 $X$ 와 인접행렬 $A$ 에 대해 연립 확률미분방정식을 세웁니다.

$$\begin{cases} dX_t = [f_{1,t}(X_t) - g_{1,t}^2 \nabla_{X_t}\log p_t(X_t,A_t)]dt + g_{1,t}d\mathbf{w}_1\\
dA_t = [f_{2,t}(A_t) - g_{2,t}^2 \nabla_{A_t}\log p_t(X_t,A_t)]dt + g_{2,t}d\mathbf{w}_2\end{cases}$$

슬라이드의 `get_sde_loss_fn` 코드에서 핵심은 세 가지입니다.
* `gen_noise(..., sym=True)` : 인접행렬 노이즈는 **대칭**이어야 함
* `mask_x / mask_adjs` : 배치 패딩된 가짜 노드를 0 으로 만듦
* DSM 손실 : `(score * std + z)^2`

## 과제

1. `vp_marginal_prob(x, t)` : VP-SDE 의 주변분포 $p(x_t\mid x_0)=\mathcal{N}(\text{mean}, \text{std}^2 I)$.
   $$\log(\text{mean coeff}) = -\tfrac14 t^2(\beta_{\max}-\beta_{\min}) - \tfrac12 t\beta_{\min},\quad
   \text{std} = \sqrt{1-e^{2\log(\text{mean coeff})}}$$
2. `gen_noise(x, flags, sym)` : `sym=True` 면 **대칭 + 대각 0** 노이즈, 아니면 노드 마스킹만.
3. `perturb_adj(adj, t, flags)` : 섭동된 인접행렬 · 사용된 노이즈 · std 를 반환.
4. `dsm_loss(score, z, std, flags)` : 마스킹을 적용한 denoising score matching 손실.
5. `SimpleGraphScore` : **순열 등변** 스코어 신경망의 `forward` 를 완성.

**힌트**: 대칭 노이즈는 `torch.tril(z, -1)` 후 자기 전치를 더하면 됩니다.
순열 등변성을 지키려면 노드 인덱스에 의존하는 연산(위치 임베딩, 첫 번째 노드만 특별 취급 등)을 쓰면 안 됩니다.

In [ ]:
BETA_MIN, BETA_MAX = 0.1, 20.0

def node_flags(adj, eps=1e-5):
    """(B, N): 패딩된 가짜 노드는 0, 실제 노드는 1."""
    return (torch.abs(adj).sum(-1) > eps).to(adj.dtype)

def mask_x(x, flags):
    return x * flags[:, :, None]

def mask_adjs(adj, flags):
    return adj * flags[:, :, None] * flags[:, None, :]


def vp_marginal_prob(x, t):
    """VP-SDE 주변분포. x: (B, N, D) 또는 (B, N, N), t: (B,)"""
    # TODO: log_mean_coeff 를 계산하고 mean, std 를 반환하세요. (브로드캐스트 축에 주의!)
    raise NotImplementedError


def gen_noise(x, flags, sym=True):
    """sym=True 면 인접행렬용 대칭 노이즈(대각 0), 아니면 노드 피처용 노이즈."""
    # TODO: torch.randn_like 로 뽑은 뒤 sym 이면 tril(-1) + 전치로 대칭화하고 마스킹하세요.
    raise NotImplementedError


def perturb_adj(adj, t, flags):
    """섭동된 인접행렬, 사용된 노이즈 z, std 를 반환."""
    # TODO: z = gen_noise(...), mean/std = vp_marginal_prob(...), perturbed = mean + std*z
    raise NotImplementedError


def dsm_loss(score, z, std, flags):
    """DSM 손실: E|| score * std + z ||^2 (패딩 노드는 제외)."""
    # TODO: (score*std + z)^2 를 만들고 마스킹한 뒤, 샘플별 합 -> 배치 평균.
    raise NotImplementedError


class SimpleGraphScore(nn.Module):
    """순열 등변(permutation equivariant) 그래프 스코어 신경망."""
    def __init__(self, in_dim=3, hid=16):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, hid)
        self.lin_neigh = nn.Linear(in_dim, hid)
        self.out_x = nn.Linear(hid, in_dim)
        self.edge = nn.Sequential(nn.Linear(hid + 1, hid), nn.SiLU(), nn.Linear(hid, 1))

    def forward(self, x, adj, flags):
        # TODO: 1) h = tanh(lin_self(x) + lin_neigh(adj @ x)) 후 마스킹
        #       2) score_x = out_x(h) 후 마스킹
        #       3) hij = h_i * h_j (대칭 결합) 와 adj 를 concat -> self.edge -> 대칭화 -> 마스킹
        raise NotImplementedError


class NaiveGraphScore(nn.Module):
    """반례: 노드 인덱스 임베딩을 쓰면 순열 등변성이 깨집니다."""
    def __init__(self, in_dim=3, hid=16, max_nodes=8):
        super().__init__()
        self.pos = nn.Embedding(max_nodes, in_dim)
        self.inner = SimpleGraphScore(in_dim, hid)

    def forward(self, x, adj, flags):
        idx = torch.arange(x.shape[1], device=x.device)
        return self.inner(x + self.pos(idx)[None], adj, flags)

In [ ]:
# ---------------- Q3 검증 ----------------
set_seed(0)

def random_graphs(B=4, N=8, n_min=5):
    """패딩이 섞인 무향 그래프 배치를 만듭니다(각 그래프는 최소한 링 구조를 가짐)."""
    adj = torch.zeros(B, N, N)
    for b in range(B):
        n = int(torch.randint(n_min, N + 1, (1,)))
        a = (torch.rand(n, n) < 0.3).float()
        a = torch.triu(a, 1)
        a = a + a.T
        ring = torch.zeros(n, n)
        for i in range(n):
            ring[i, (i + 1) % n] = 1.0
        a = ((a + ring + ring.T) > 0).float()
        adj[b, :n, :n] = a
    flags = node_flags(adj)
    x = mask_x(torch.randn(B, N, 3), flags)
    return x, adj, flags

X, A, FL = random_graphs()
B, N = A.shape[0], A.shape[1]
t = torch.rand(B) * 0.9 + 0.05

# (1) VP-SDE
mean, std = vp_marginal_prob(A, t)
assert mean.shape == A.shape and std.shape == (B,)
t_small, t_big = torch.tensor([0.01]), torch.tensor([1.0])
_, s_small = vp_marginal_prob(A[:1], t_small)
_, s_big = vp_marginal_prob(A[:1], t_big)
assert s_small < s_big, "t 가 커질수록 std 가 증가해야 합니다."
m_big, _ = vp_marginal_prob(A[:1], t_big)
assert m_big.abs().max() < A[:1].abs().max(), "t 가 커질수록 평균은 0 으로 줄어들어야 합니다."

# (2) 대칭 노이즈
z = gen_noise(A, FL, sym=True)
assert torch.allclose(z, z.transpose(-1, -2), atol=ATOL), "인접행렬 노이즈는 대칭이어야 합니다."
assert torch.allclose(torch.diagonal(z, dim1=-2, dim2=-1), torch.zeros(B, N), atol=ATOL), "자기 자신 간선은 없어야 합니다."
assert torch.allclose(z, mask_adjs(z, FL), atol=ATOL), "패딩 노드는 0 이어야 합니다."
zx = gen_noise(X, FL, sym=False)
assert torch.allclose(zx, mask_x(zx, FL), atol=ATOL)

# (3) 섭동
Ap, z, std = perturb_adj(A, t, FL)
assert torch.allclose(Ap, Ap.transpose(-1, -2), atol=ATOL), "섭동 후에도 대칭이어야 합니다."
assert torch.allclose(Ap, mask_adjs(Ap, FL), atol=ATOL)

# (4) 최적 스코어 -> 손실 0
optimal_score = -z / std[:, None, None]
assert dsm_loss(optimal_score, z, std, FL).item() < 1e-6, "최적 스코어 -z/std 에서 손실은 0 이어야 합니다."
assert dsm_loss(torch.zeros_like(z), z, std, FL).item() > 1.0, "영 스코어의 손실은 커야 합니다."

# (5) 순열 등변성
set_seed(3)
model = SimpleGraphScore(in_dim=3, hid=16)
perm = torch.randperm(N)
sx1, sa1 = model(X, A, FL)
sx2, sa2 = model(X[:, perm], A[:, perm][:, :, perm], FL[:, perm])
assert sa1.shape == A.shape and sx1.shape == X.shape
assert torch.allclose(sa1, sa1.transpose(-1, -2), atol=ATOL), "인접행렬 스코어도 대칭이어야 합니다."
assert torch.allclose(sx2, sx1[:, perm], atol=1e-5), "노드 스코어가 순열 등변이 아닙니다."
assert torch.allclose(sa2, sa1[:, perm][:, :, perm], atol=1e-5), "간선 스코어가 순열 등변이 아닙니다."

# (6) 반례 모델은 등변성이 깨져야 정상
naive = NaiveGraphScore(in_dim=3, hid=16, max_nodes=N)
nx1, na1 = naive(X, A, FL)
nx2, na2 = naive(X[:, perm], A[:, perm][:, :, perm], FL[:, perm])
assert not torch.allclose(na2, na1[:, perm][:, :, perm], atol=1e-6), "반례 모델은 등변성이 깨져야 합니다."

print("✅ Q3 통과")

---
# Q4. GeoDiff: SE(3) 불변 메시지 전달과 등변 좌표 업데이트 (16점)

## 배경

분자 3차원 구조를 생성할 때, **분자를 회전·평행이동해도 같은 분자**입니다.
GeoDiff 는 슬라이드의 식처럼 **거리만 사용**하여 피처를 갱신합니다.

$$m_{ij} = \Phi_m\big(h_i^l, h_j^l, \|x_i^l - x_j^l\|^2, e_{ij};\theta_m\big)$$
$$h_i^{l+1} = \Phi_h\Big(h_i^l, \sum_{j\in N(i)} m_{ij}; \theta_h\Big)$$
$$x_i^{l+1} = \sum_{j\in N(i)} \frac{1}{d_{ij}}(c_i - c_j)\,\Phi_x(m_{ij};\theta_x)$$

* $h$ 는 거리에만 의존 → **회전·병진 불변(invariant)**
* $x$ 업데이트는 방향 벡터 $(c_i-c_j)$ 에 스칼라를 곱한 합 → **회전 등변(equivariant), 병진 불변**

## 과제

1. `gaussian_smearing(d, ...)` : 거리를 가우시안 기저로 펼치는 엣지 피처(SchNet 의 `edge_attr`).
2. `cosine_cutoff(d, cutoff, smooth)` : 슬라이드 `CFConv` 의 $C$ 계산.
   $C = 0.5(\cos(d\pi/\text{cutoff})+1)$ 이고 $0\le d\le \text{cutoff}$ 밖에서는 0.
3. `CFConv.forward` : $W = \text{mlp}(e_{ij})\cdot C$, 메시지 $= h_{src}\odot W$ 를 목적지 노드로 합산.
4. `EquivariantCoordUpdate.forward` : 위 세 번째 식을 구현.

**힌트**: 인덱스 합산은 `torch.zeros_like(h).index_add(0, dst, msg)` 를 쓰세요.
좌표 업데이트에서 `pos` 를 **직접** 신경망에 넣으면 등변성이 깨집니다. 오직 차이 벡터와 거리만 사용하세요.

In [ ]:
class ShiftedSoftplus(nn.Module):
    def forward(self, x):
        return F.softplus(x) - math.log(2.0)


def gaussian_smearing(d, num_gaussians=16, cutoff=10.0):
    """거리 d: (E,) -> (E, num_gaussians) 가우시안 기저."""
    offset = torch.linspace(0.0, cutoff, num_gaussians)
    coeff = -0.5 / (offset[1] - offset[0]).item() ** 2
    # TODO: exp(coeff * (d - offset)^2) 형태로 브로드캐스트해 (E, num_gaussians) 를 만드세요.
    raise NotImplementedError


def cosine_cutoff(d, cutoff=10.0, smooth=True):
    """부드러운 컷오프 계수 C: (E,)"""
    # TODO: smooth 면 0.5*(cos(d*pi/cutoff)+1) 에 0 <= d <= cutoff 마스크를 곱하고,
    #       아니면 (d <= cutoff) 를 float 로 반환하세요.
    raise NotImplementedError


class CFConv(nn.Module):
    """SchNet 연속 필터 합성곱 (슬라이드 CFConv 대응)."""
    def __init__(self, in_channels, out_channels, num_filters, mlp, cutoff=10.0, smooth=True):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, num_filters, bias=False)
        self.lin2 = nn.Linear(num_filters, out_channels)
        self.mlp, self.cutoff, self.smooth = mlp, cutoff, smooth

    def forward(self, h, edge_index, edge_length, edge_attr):
        src, dst = edge_index[0], edge_index[1]
        # TODO: 1) C = cosine_cutoff(...)  2) W = self.mlp(edge_attr) * C.view(-1,1)
        #       3) h = self.lin1(h),  msg = h[src] * W
        #       4) index_add 로 dst 별 합산 후 self.lin2 통과
        raise NotImplementedError


class InteractionBlock(nn.Module):
    def __init__(self, hidden_channels, num_gaussians, num_filters, cutoff=10.0, smooth=True):
        super().__init__()
        mlp = nn.Sequential(nn.Linear(num_gaussians, num_filters), ShiftedSoftplus(),
                            nn.Linear(num_filters, num_filters))
        self.conv = CFConv(hidden_channels, hidden_channels, num_filters, mlp, cutoff, smooth)
        self.act = ShiftedSoftplus()
        self.lin = nn.Linear(hidden_channels, hidden_channels)

    def forward(self, h, edge_index, edge_length, edge_attr):
        h = self.conv(h, edge_index, edge_length, edge_attr)
        return self.lin(self.act(h))


class SchNetEncoder(nn.Module):
    """슬라이드의 SchNetEncoder 축약판: 거리만 쓰므로 SE(3) 불변."""
    def __init__(self, hidden_channels=32, num_filters=32, num_gaussians=16,
                 num_interactions=2, cutoff=10.0, smooth=True):
        super().__init__()
        self.embedding = nn.Embedding(100, hidden_channels)
        self.interactions = nn.ModuleList([
            InteractionBlock(hidden_channels, num_gaussians, num_filters, cutoff, smooth)
            for _ in range(num_interactions)])

    def forward(self, z, edge_index, edge_length, edge_attr):
        h = self.embedding(z)
        for block in self.interactions:
            h = h + block(h, edge_index, edge_length, edge_attr)
        return h


class EquivariantCoordUpdate(nn.Module):
    """x_i <- sum_j (1/d_ij) (c_i - c_j) * Phi_x(m_ij)"""
    def __init__(self, hidden_channels=32, num_gaussians=16):
        super().__init__()
        self.phi_m = nn.Sequential(nn.Linear(2 * hidden_channels + num_gaussians, hidden_channels),
                                   ShiftedSoftplus(), nn.Linear(hidden_channels, hidden_channels))
        self.phi_x = nn.Sequential(nn.Linear(hidden_channels, hidden_channels),
                                   ShiftedSoftplus(), nn.Linear(hidden_channels, 1))

    def forward(self, h, pos, edge_index, edge_length, edge_attr):
        src, dst = edge_index[0], edge_index[1]
        # TODO: 1) m = phi_m([h_dst, h_src, edge_attr])  2) w = phi_x(m)  (E,1)
        #       3) diff = pos[dst] - pos[src] 를 거리로 나눈 단위벡터에 w 를 곱하고
        #       4) dst 기준 index_add 로 합산해 (N, 3) 업데이트를 반환
        raise NotImplementedError


class NonEquivariantUpdate(nn.Module):
    """반례: 좌표를 그대로 신경망에 넣으면 회전 등변성이 깨집니다."""
    def __init__(self, hidden_channels=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hidden_channels + 3, hidden_channels),
                                 ShiftedSoftplus(), nn.Linear(hidden_channels, 3))

    def forward(self, h, pos, edge_index, edge_length, edge_attr):
        return self.net(torch.cat([h, pos], dim=-1))

In [ ]:
# ---------------- Q4 검증 ----------------
set_seed(0)
CUTOFF, NG = 10.0, 16

def build_edges(pos, cutoff=CUTOFF):
    """컷오프 안의 모든 원자쌍을 간선으로. edge_index = [src, dst]."""
    n = pos.shape[0]
    d = torch.cdist(pos, pos)
    m = (d <= cutoff) & (~torch.eye(n, dtype=torch.bool))
    edge_index = m.nonzero().T.contiguous()
    edge_length = d[edge_index[0], edge_index[1]]
    return edge_index, edge_length

def random_rotation():
    q, r = torch.linalg.qr(torch.randn(3, 3))
    q = q * torch.sign(torch.diagonal(r))[None, :]
    if torch.det(q) < 0:
        q = q * torch.tensor([[-1.0, 1.0, 1.0]])
    return q

n_atoms = 10
pos = torch.randn(n_atoms, 3) * 1.2          # 모든 원자쌍이 컷오프(10) 안에 들어오도록
z = torch.randint(1, 10, (n_atoms,))
R, tr = random_rotation(), torch.randn(3) * 3.0
pos2 = pos @ R.T + tr                                   # 회전 + 평행이동

assert torch.allclose(R @ R.T, torch.eye(3), atol=1e-5) and torch.det(R) > 0

ei, el = build_edges(pos)
ei2, el2 = build_edges(pos2)
assert torch.equal(ei, ei2), "거리가 보존되므로 간선 집합도 같아야 합니다."
assert torch.allclose(el, el2, atol=1e-4), "회전·병진은 원자 간 거리를 바꾸지 않습니다."

ea, ea2 = gaussian_smearing(el, NG, CUTOFF), gaussian_smearing(el2, NG, CUTOFF)
assert ea.shape == (el.shape[0], NG)
assert torch.allclose(ea, ea2, atol=1e-4)
assert torch.all(cosine_cutoff(torch.tensor([CUTOFF + 1.0]), CUTOFF) == 0.0), "컷오프 밖은 0"
assert torch.allclose(cosine_cutoff(torch.tensor([0.0]), CUTOFF), torch.tensor([1.0]), atol=1e-5)

# (1) 노드 피처의 SE(3) 불변성
set_seed(7)
enc = SchNetEncoder(hidden_channels=32, num_filters=32, num_gaussians=NG, num_interactions=2, cutoff=CUTOFF)
with torch.no_grad():
    h1 = enc(z, ei, el, ea)
    h2 = enc(z, ei2, el2, ea2)
assert h1.shape == (n_atoms, 32)
assert torch.allclose(h1, h2, atol=1e-4), "SchNet 피처는 회전·병진에 불변이어야 합니다."

# (2) 좌표 업데이트의 회전 등변성 / 병진 불변성
set_seed(8)
upd = EquivariantCoordUpdate(hidden_channels=32, num_gaussians=NG)
with torch.no_grad():
    u1 = upd(h1, pos, ei, el, ea)
    u2 = upd(h2, pos2, ei2, el2, ea2)
assert u1.shape == (n_atoms, 3)
assert torch.allclose(u2, u1 @ R.T, atol=1e-4), "좌표 업데이트는 회전 등변이어야 합니다 (u(Rx+t) = R u(x))."

pos_tr = pos + torch.tensor([1.0, -2.0, 3.0])
ei_tr, el_tr = build_edges(pos_tr)
with torch.no_grad():
    u_tr = upd(h1, pos_tr, ei_tr, el_tr, gaussian_smearing(el_tr, NG, CUTOFF))
assert torch.allclose(u_tr, u1, atol=1e-4), "평행이동만 하면 업데이트가 변하지 않아야 합니다."

# (3) 노드 순서(치환) 등변성
perm = torch.randperm(n_atoms)
inv = torch.argsort(perm)
pos_p, z_p = pos[perm], z[perm]
ei_p, el_p = build_edges(pos_p)
ea_p = gaussian_smearing(el_p, NG, CUTOFF)
with torch.no_grad():
    h_p = enc(z_p, ei_p, el_p, ea_p)
    u_p = upd(h_p, pos_p, ei_p, el_p, ea_p)
assert torch.allclose(h_p, h1[perm], atol=1e-4), "노드 순서를 바꾸면 피처도 같은 순서로 바뀌어야 합니다."
assert torch.allclose(u_p, u1[perm], atol=1e-4)

# (4) 반례 모델은 등변성이 깨져야 정상
set_seed(9)
bad = NonEquivariantUpdate(hidden_channels=32)
with torch.no_grad():
    b1 = bad(h1, pos, ei, el, ea)
    b2 = bad(h2, pos2, ei2, el2, ea2)
assert not torch.allclose(b2, b1 @ R.T, atol=1e-5), "반례 모델은 등변성이 깨져야 합니다."

print("✅ Q4 통과")

---
# Q5. 리만 다양체 위의 확산 (구면 $S^2$) (12점)

## 배경

슬라이드의 "스트리밍(매니폴드) 구조의 데이터 — 리만 스코어 기반 생성 모델" 부분입니다.
지구 표면의 지진 좌표, 분자의 이면각, 회전 행렬처럼 데이터가 **곡면 위에서만** 정의되는 경우
유클리드 노이즈를 더하면 데이터가 다양체 밖으로 튀어나갑니다.

리만 확산은 각 점의 **접공간(tangent space)** 에서 노이즈를 뽑고 **지수 사상(exponential map)** 으로
다양체 위를 걷습니다. 단위 구면 $S^2$ 에서는

$$P_x(v) = v - \langle v, x\rangle x, \qquad
\exp_x(v) = \cos(\|v\|)\,x + \sin(\|v\|)\,\frac{v}{\|v\|}$$

## 과제

1. `project_tangent(x, v)` : 접공간 투영.
2. `exp_map(x, v)` : 구면 위 지수 사상.
3. `geodesic_distance(x, y)` : $\arccos(\langle x,y\rangle)$ (수치 안정성 주의).
4. `riemannian_forward(x0, sigmas)` : 매 스텝 접공간 노이즈 → 지수 사상으로 이동하는 순방향 확산.
5. `euclidean_forward(x0, sigmas)` : 비교용. 단순히 $x \leftarrow x + \sigma z$.

**힌트**: $\|v\|$ 가 0 에 가까울 때 나눗셈이 터지지 않도록 `clamp_min` 을 쓰세요.
`arccos` 의 입력은 `clamp(-1+eps, 1-eps)` 로 잘라야 합니다.

In [ ]:
def project_tangent(x, v):
    """점 x 에서의 접공간으로 v 를 투영. x: (..., 3) 단위벡터."""
    # TODO: v 에서 x 방향 성분을 빼세요.
    raise NotImplementedError


def exp_map(x, v, eps=1e-12):
    """구면 위 지수 사상. v 는 x 의 접벡터."""
    # TODO: cos(||v||) x + sin(||v||) v/||v||
    raise NotImplementedError


def geodesic_distance(x, y, eps=1e-6):
    """구면 위 두 점 사이의 측지 거리(호의 길이)."""
    # TODO: acos(<x,y>) — 입력을 clamp 하세요.
    raise NotImplementedError


def riemannian_forward(x0, sigmas):
    """매 스텝 접공간 가우시안 -> 지수 사상. 반환 (T+1, B, 3)."""
    # TODO: z = project_tangent(x, randn), x = exp_map(x, sigma * z) 를 반복하세요.
    raise NotImplementedError


def euclidean_forward(x0, sigmas):
    """비교용: 다양체를 무시한 유클리드 확산."""
    # TODO: x = x + sigma * randn 을 반복하세요.
    raise NotImplementedError

In [ ]:
# ---------------- Q5 검증 ----------------
set_seed(0)
B = 4000
x0 = torch.zeros(B, 3)
x0[:, 2] = 1.0                                  # 모두 북극에서 출발

# (1) 접공간 투영
v = torch.randn(B, 3)
vt = project_tangent(x0, v)
assert torch.allclose((vt * x0).sum(-1), torch.zeros(B), atol=1e-5), "접벡터는 x 와 직교해야 합니다."

# (2) 지수 사상은 구면 위에 머무름
y = exp_map(x0, 0.7 * vt / vt.norm(dim=-1, keepdim=True))
assert torch.allclose(y.norm(dim=-1), torch.ones(B), atol=1e-5), "exp_map 결과는 단위벡터여야 합니다."
assert torch.allclose(geodesic_distance(x0, y), torch.full((B,), 0.7), atol=1e-3), \
    "이동 거리는 접벡터의 길이와 같아야 합니다."
assert geodesic_distance(x0, x0).max() < 1e-2

# (3) 리만 확산은 항상 구면 위
sigmas = torch.full((500,), 0.15)
traj = riemannian_forward(x0, sigmas)
assert traj.shape == (len(sigmas) + 1, B, 3)
norms = traj.norm(dim=-1)
assert (norms - 1.0).abs().max() < 1e-4, "모든 시점에서 노름이 1 이어야 합니다."

# (4) 충분히 확산되면 구면 균등분포로 수렴
final = traj[-1]
assert final.mean(0).norm() < 0.12, f"균등분포라면 평균이 0 에 가까워야 합니다: {final.mean(0).norm():.3f}"
zc = final[:, 2]                                 # 아르키메데스 정리: z ~ Uniform(-1, 1)
assert abs(zc.mean().item()) < 0.08
assert abs(zc.var().item() - 1.0 / 3.0) < 0.06, f"z 성분 분산이 1/3 이어야 합니다: {zc.var():.3f}"

# (5) 유클리드 확산은 다양체를 벗어남
eu = euclidean_forward(x0, sigmas)
eu_norm_err = (eu[-1].norm(dim=-1) - 1.0).abs().mean()
assert eu_norm_err > 1.0, "유클리드 확산은 구면을 크게 벗어나야 합니다."
print(f"리만 확산 노름 오차 = {(norms[-1]-1).abs().mean():.2e} | 유클리드 확산 노름 오차 = {eu_norm_err:.3f}")
print(f"최종 z 성분: 평균 {zc.mean():.4f}, 분산 {zc.var():.4f} (균등분포 이론값 0, 0.3333)")

print("✅ Q5 통과")

---
# Q6. 잠재 확산 모델 (Stable Diffusion 축약판) (16점)

## 배경

슬라이드 마지막 부분의 `AutoencoderKL` / `DiagonalGaussianDistribution` 코드에 해당합니다.
다양체 구조를 **모를 때**의 해법은 "오토인코더로 저차원 잠재공간을 학습한 뒤 그 안에서 확산"입니다.

$$x \xrightarrow{\ \mathcal{E}\ } q(z\mid x)=\mathcal{N}(\mu(x), \sigma^2(x)) \xrightarrow{\ \text{DDPM}\ } z \xrightarrow{\ \mathcal{D}\ } \hat{x}$$

## 과제

1. `DiagonalGaussianDistribution` : `sample()`, `mode()`, `kl()` 를 완성.
   KL 은 표준정규와의 KL: $\frac12\sum(\mu^2 + \sigma^2 - 1 - \log\sigma^2)$.
2. `TinyAutoencoderKL.forward` : 인코딩 → 샘플링 → 디코딩.
3. `ddpm_loss(eps_model, z0, ...)` : 잠재공간 DDPM 의 $\epsilon$-예측 손실.
4. 학습 루프를 돌려 손실이 감소하는지 확인.

**힌트**: `torch.chunk(parameters, 2, dim=1)` 로 평균/로그분산을 나눕니다.
`logvar` 는 `clamp(-30, 20)` 으로 잘라야 학습이 안정적입니다.

In [ ]:
class DiagonalGaussianDistribution:
    """슬라이드의 DiagonalGaussianDistribution 대응."""
    def __init__(self, parameters, deterministic=False):
        self.parameters = parameters
        # TODO: chunk 로 mean, logvar 분리 -> logvar clamp(-30, 20) -> std, var 계산
        #       deterministic 이면 std = var = 0
        raise NotImplementedError

    def sample(self):
        # TODO: reparameterization trick
        raise NotImplementedError

    def mode(self):
        # TODO
        raise NotImplementedError

    def kl(self):
        """KL( N(mean, var) || N(0, I) ), 샘플별 스칼라."""
        # TODO: 0.5 * sum(mean^2 + var - 1 - logvar) (채널·공간 축 합)
        raise NotImplementedError


class TinyAutoencoderKL(nn.Module):
    """32x32 -> 8x8 로 줄이는 초소형 AutoencoderKL."""
    def __init__(self, z_ch=4):
        super().__init__()
        self.z_ch = z_ch
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(32, 2 * z_ch, 3, padding=1),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(z_ch, 32, 3, padding=1), nn.SiLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.SiLU(),
            nn.ConvTranspose2d(16, 16, 4, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(16, 1, 3, padding=1),
        )

    def encode(self, x):
        return DiagonalGaussianDistribution(self.encoder(x))

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, sample_posterior=True):
        # TODO: encode -> (sample_posterior 면 sample(), 아니면 mode()) -> decode
        #       (재구성, posterior) 튜플을 반환하세요.
        raise NotImplementedError


class TinyEps(nn.Module):
    """잠재공간 DDPM 의 노이즈 예측기."""
    def __init__(self, z_ch=4, hid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(z_ch + 1, hid, 3, padding=1), nn.SiLU(),
            nn.Conv2d(hid, hid, 3, padding=1), nn.SiLU(),
            nn.Conv2d(hid, z_ch, 3, padding=1),
        )

    def forward(self, z, t_norm):
        tt = t_norm.view(-1, 1, 1, 1).expand(-1, 1, z.shape[2], z.shape[3])
        return self.net(torch.cat([z, tt], dim=1))


def ddpm_loss(eps_model, z0, alphas_bar):
    """잠재공간 epsilon-예측 손실."""
    # TODO: 1) t 를 균등 샘플링, ab = alphas_bar[t] 를 (B,1,1,1) 로 reshape
    #       2) zt = sqrt(ab) z0 + sqrt(1-ab) eps
    #       3) eps_model(zt, t/T) 와 eps 의 MSE
    raise NotImplementedError

In [ ]:
# ---------------- Q6 검증 ----------------
set_seed(0)

# (0) 합성 데이터: 32x32 흑백 도형
def make_shapes(n, size=32):
    imgs = torch.zeros(n, 1, size, size)
    yy, xx = torch.meshgrid(torch.arange(size).float(), torch.arange(size).float(), indexing="ij")
    for i in range(n):
        h = int(torch.randint(8, 16, (1,)))
        r0 = int(torch.randint(0, size - h, (1,)))
        c0 = int(torch.randint(0, size - h, (1,)))
        if i % 2 == 0:
            imgs[i, 0, r0:r0 + h, c0:c0 + h] = 1.0
        else:
            cy, cx, rad = r0 + h / 2.0, c0 + h / 2.0, h / 2.0
            imgs[i, 0] = (((yy - cy) ** 2 + (xx - cx) ** 2) <= rad ** 2).float()
    return imgs * 2.0 - 1.0

data = make_shapes(512)

# (1) DiagonalGaussianDistribution 성질
params = torch.zeros(3, 8, 4, 4)                       # mean=0, logvar=0 -> 표준정규
dist = DiagonalGaussianDistribution(params)
assert dist.mean.shape == (3, 4, 4, 4) and dist.logvar.shape == (3, 4, 4, 4)
assert torch.allclose(dist.kl(), torch.zeros(3), atol=1e-6), "표준정규와의 KL 은 0 이어야 합니다."

params2 = torch.cat([torch.full((2, 4, 4, 4), 2.0), torch.full((2, 4, 4, 4), math.log(0.25))], dim=1)
kl_expected = 0.5 * (4.0 + 0.25 - 1.0 - math.log(0.25)) * (4 * 4 * 4)
assert torch.allclose(DiagonalGaussianDistribution(params2).kl(),
                      torch.full((2,), kl_expected), rtol=1e-4), "KL 공식을 확인하세요."

det = DiagonalGaussianDistribution(torch.randn(2, 8, 4, 4), deterministic=True)
assert torch.allclose(det.sample(), det.mode()), "deterministic=True 면 샘플이 곧 평균입니다."
assert torch.allclose(det.kl(), torch.zeros(2))

s = DiagonalGaussianDistribution(torch.cat([torch.zeros(4096, 2, 2, 2),
                                            torch.zeros(4096, 2, 2, 2)], dim=1)).sample()
assert abs(s.mean().item()) < 0.05 and abs(s.std().item() - 1.0) < 0.05, "reparameterization 을 확인하세요."

# (2) 오토인코더 학습 (CPU 약 1분)
set_seed(1)
ae = TinyAutoencoderKL(z_ch=4)
opt = torch.optim.Adam(ae.parameters(), lr=2e-3)
KL_W, STEPS, BS = 1e-6, 600, 32
hist = []
t_start = time.time()
for step in range(STEPS):
    idx = torch.randint(0, data.shape[0], (BS,))
    x = data[idx]
    rec, post = ae(x)
    loss = F.mse_loss(rec, x) + KL_W * post.kl().mean()
    opt.zero_grad(); loss.backward(); opt.step()
    hist.append(F.mse_loss(rec, x).item())
print(f"AE 학습 {time.time()-t_start:.1f}s | 초기 재구성 MSE {np.mean(hist[:20]):.4f} -> 최종 {np.mean(hist[-20:]):.4f}")
assert np.mean(hist[-20:]) < np.mean(hist[:20]) * 0.5, "재구성 손실이 충분히 감소해야 합니다."

with torch.no_grad():
    z_all = ae.encode(data[:256]).mode()
assert z_all.shape == (256, 4, 8, 8)
print(f"픽셀 차원 {32*32*1} -> 잠재 차원 {4*8*8} (압축률 {32*32/(4*8*8):.1f}x)")

# (3) 잠재공간 DDPM
set_seed(2)
z_scale = z_all.std()
z_data = (z_all / z_scale).detach()                    # 잠재값 정규화 (Stable Diffusion 의 scale_factor)
T = 200
betas_d = torch.linspace(1e-4, 0.02, T)
alphas_bar = torch.cumprod(1.0 - betas_d, dim=0)

eps_model = TinyEps(z_ch=4, hid=64)
opt2 = torch.optim.Adam(eps_model.parameters(), lr=2e-3)
hist2 = []
for step in range(300):
    idx = torch.randint(0, z_data.shape[0], (BS,))
    loss = ddpm_loss(eps_model, z_data[idx], alphas_bar)
    opt2.zero_grad(); loss.backward(); opt2.step()
    hist2.append(loss.item())
print(f"잠재 DDPM 손실 {np.mean(hist2[:20]):.4f} -> {np.mean(hist2[-20:]):.4f}")
assert np.mean(hist2[-20:]) < np.mean(hist2[:20]), "DDPM 손실이 감소해야 합니다."
assert 0.0 < np.mean(hist2[-20:]) < 1.5

# (4) 역과정으로 잠재 샘플 -> 디코딩 (형태 확인)
@torch.no_grad()
def ddpm_sample(eps_model, shape, betas_d, alphas_bar):
    z = torch.randn(shape)
    alphas = 1.0 - betas_d
    for i in reversed(range(len(betas_d))):
        t = torch.full((shape[0],), i, dtype=torch.float32) / len(betas_d)
        eps_hat = eps_model(z, t)
        mean = (z - betas_d[i] / (1 - alphas_bar[i]).sqrt() * eps_hat) / alphas[i].sqrt()
        z = mean + (betas_d[i].sqrt() * torch.randn_like(z) if i > 0 else 0.0)
    return z

z_gen = ddpm_sample(eps_model, (4, 4, 8, 8), betas_d, alphas_bar)
with torch.no_grad():
    x_gen = ae.decode(z_gen * z_scale)
assert x_gen.shape == (4, 1, 32, 32) and torch.isfinite(x_gen).all()

print("✅ Q6 통과")

---
# Q7. 크로스 어텐션과 분류기 없는 안내(CFG) (12점)

## 배경

슬라이드 마지막 코드인 Stable Diffusion 의 `CrossAttention` 입니다.
잠재 이미지 토큰이 **쿼리**, 텍스트 임베딩이 **키/값** 이 되어 조건 정보를 주입합니다.

$$\text{Attention}(Q,K,V) = \mathrm{softmax}\Big(\frac{QK^\top}{\sqrt{d}}\Big)V,\qquad
Q = W_q x,\; K = W_k c,\; V = W_v c$$

## 과제

1. `CrossAttention.forward` 를 완성하세요.
   * 멀티헤드 분리 `(B, N, H*D) -> (B*H, N, D)`
   * `sim = einsum('b i d, b j d -> b i j') * scale`
   * `mask` 가 주어지면 해당 위치를 $-\infty$ 로 채우기
   * softmax → 값 가중합 → 헤드 병합 → `to_out`
2. `classifier_free_guidance(eps_uncond, eps_cond, w)` 구현.

**힌트**: `context=None` 이면 self-attention 이 됩니다.
마스크는 `(B, M)` 불리언이며, `repeat_interleave(heads, dim=0)` 로 헤드 축에 맞춰 늘려야 합니다.

In [ ]:
class CrossAttention(nn.Module):
    """Stable Diffusion 의 CrossAttention 축약판."""
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64, dropout=0.0):
        super().__init__()
        inner_dim = dim_head * heads
        context_dim = query_dim if context_dim is None else context_dim
        self.heads, self.dim_head = heads, dim_head
        self.scale = dim_head ** -0.5
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner_dim, query_dim), nn.Dropout(dropout))

    def _split_heads(self, t):
        """(B, N, H*D) -> (B*H, N, D)  (배치가 바깥, 헤드가 안쪽 순서)"""
        b, n, _ = t.shape
        h = self.heads
        return t.reshape(b, n, h, self.dim_head).permute(0, 2, 1, 3).reshape(b * h, n, self.dim_head)

    def _merge_heads(self, t, b):
        """(B*H, N, D) -> (B, N, H*D)"""
        h = self.heads
        n = t.shape[1]
        return t.reshape(b, h, n, self.dim_head).permute(0, 2, 1, 3).reshape(b, n, h * self.dim_head)

    def forward(self, x, context=None, mask=None, return_attn=False):
        b, h = x.shape[0], self.heads
        # TODO: 1) context 가 None 이면 x 로 대체 (self-attention)
        #       2) q, k, v 를 만들고 _split_heads 로 헤드 분리
        #       3) sim = einsum("b i d, b j d -> b i j", q, k) * self.scale
        #       4) mask 가 있으면 masked_fill 로 -finfo.max 채우기
        #       5) softmax -> einsum 으로 v 가중합 -> _merge_heads -> to_out
        #       return_attn 이 True 면 (출력, attn) 을 반환하세요.
        raise NotImplementedError


def classifier_free_guidance(eps_uncond, eps_cond, w):
    """eps = eps_uncond + w * (eps_cond - eps_uncond)"""
    # TODO
    raise NotImplementedError

In [ ]:
# ---------------- Q7 검증 ----------------
set_seed(0)
B, N, M = 3, 16, 7                       # 배치 / 이미지 토큰 수 / 텍스트 토큰 수
QD, CD, H, DH = 32, 24, 4, 8

attn_layer = CrossAttention(query_dim=QD, context_dim=CD, heads=H, dim_head=DH).eval()
x = torch.randn(B, N, QD)
ctx = torch.randn(B, M, CD)

out, A = attn_layer(x, context=ctx, return_attn=True)
assert out.shape == (B, N, QD), f"출력 shape 이 잘못되었습니다: {out.shape}"
assert A.shape == (B * H, N, M), f"어텐션 shape 이 잘못되었습니다: {A.shape}"
assert torch.allclose(A.sum(-1), torch.ones(B * H, N), atol=1e-5), "어텐션 가중치의 행 합은 1"
assert torch.all(A >= 0)

# 마스킹: 뒤쪽 3개 텍스트 토큰은 패딩이라고 가정
mask = torch.ones(B, M, dtype=torch.bool)
mask[:, -3:] = False
out_m, A_m = attn_layer(x, context=ctx, mask=mask, return_attn=True)
assert A_m[:, :, -3:].abs().max() < 1e-6, "마스킹된 토큰의 어텐션은 0 이어야 합니다."
assert torch.allclose(A_m.sum(-1), torch.ones(B * H, N), atol=1e-5)
assert not torch.allclose(out_m, out, atol=1e-6), "마스킹은 출력에 영향을 주어야 합니다."

# 배치별로 마스크가 독립적으로 적용되는지 (헤드 축 확장 확인)
mask2 = torch.ones(B, M, dtype=torch.bool)
mask2[0, -3:] = False
_, A_2 = attn_layer(x, context=ctx, mask=mask2, return_attn=True)
assert A_2[:H, :, -3:].abs().max() < 1e-6, "0번 샘플만 마스킹되어야 합니다."
assert A_2[H:, :, -3:].abs().max() > 1e-6, "1,2번 샘플은 마스킹되면 안 됩니다. (repeat_interleave 순서 확인)"

# self-attention 동치성
self_layer = CrossAttention(query_dim=QD, context_dim=None, heads=H, dim_head=DH).eval()
assert torch.allclose(self_layer(x), self_layer(x, context=x), atol=1e-6), "context=None 은 self-attention 입니다."

# 조건이 바뀌면 출력도 바뀐다
ctx_b = torch.randn(B, M, CD)
assert not torch.allclose(attn_layer(x, ctx), attn_layer(x, ctx_b), atol=1e-6), "조건이 출력에 반영되어야 합니다."

# 헤드 분리/병합이 항등인지 (구현 실수 방지)
t = torch.randn(B, N, H * DH)
assert torch.allclose(attn_layer._merge_heads(attn_layer._split_heads(t), B), t, atol=1e-6)

# CFG
eu, ec = torch.randn(2, 4, 4), torch.randn(2, 4, 4)
assert torch.allclose(classifier_free_guidance(eu, ec, 0.0), eu, atol=1e-6)
assert torch.allclose(classifier_free_guidance(eu, ec, 1.0), ec, atol=1e-6)
assert torch.allclose(classifier_free_guidance(eu, ec, 7.5), eu + 7.5 * (ec - eu), atol=1e-6)
w_big = classifier_free_guidance(eu, ec, 7.5)
assert (w_big - ec).norm() > (ec - eu).norm(), "w > 1 이면 조건 방향으로 더 밀어냅니다."

print("✅ Q7 통과")

---
# 서술형 문제 (제출용, 각 3~5줄)

아래 셀 아래에 마크다운 셀을 만들어 답을 작성하세요.

1. **(Q1–Q2)** 연속 확산의 $\mathcal{N}(0,I)$ 사전분포에 대응하는 것이, 균등 커널과 마스크 흡수 커널에서는 각각 무엇인가?
   두 커널 중 텍스트-투-이미지 토큰 생성에 마스크 흡수 커널이 더 적합한 이유를 설명하시오.
2. **(Q3)** 인접행렬 노이즈를 대칭화하지 않으면 학습·생성에서 구체적으로 어떤 문제가 생기는가?
   슬라이드 그림 5-2 의 "Node-Edge Mismatch" 와 연결해 설명하시오.
3. **(Q4)** 확산 모델이 예측하는 $\epsilon$ 은 왜 회전 **불변**이 아니라 **등변**이어야 하는가?
   등변성을 데이터 증강(회전된 샘플 추가)으로 대체할 때의 장단점은?
4. **(Q5–Q6)** 데이터의 다양체 구조를 "알고 있는 경우"(리만 확산)와 "모르는 경우"(잠재 확산)의
   장단점을 비교하시오. 단백질 구조 생성이라면 어느 쪽을 택하겠는가?
5. **(Q7)** CFG 가중치 $w$ 를 1 → 15 로 올릴 때 생성 결과의 프롬프트 충실도·다양성·화질이
   각각 어떻게 변할지 예측하고 그 이유를 설명하시오.

---
## 제출 방법

1. 모든 검증 셀에서 `✅ Qn 통과` 가 출력되는지 확인합니다.
2. 서술형 답안을 마크다운 셀에 작성합니다.
3. `파일 > .ipynb 다운로드` 로 저장 후 `학번_이름_3주차.ipynb` 로 제출합니다.

## 더 해보기 (가산점)

* Q1 의 균등 커널과 Q2 의 마스크 커널로 **같은 토이 데이터**를 학습시켜 샘플 품질을 비교해 보세요.
* Q3 의 스코어 모델을 실제로 학습시켜 작은 그래프(예: 링, 격자)를 생성해 보세요.
* Q6 의 오토인코더를 **결정적**(KL 없음)으로 바꾸면 잠재 DDPM 샘플이 어떻게 달라지는지 확인해 보세요.
* Q7 의 크로스 어텐션을 Q6 의 `TinyEps` 에 붙여, "사각형 / 원" 두 클래스 조건부 생성 + CFG 를 구현해 보세요.

## 참고 문헌

* Gu et al., *Vector Quantized Diffusion Model for Text-to-Image Synthesis*, CVPR 2022
* Campbell et al., *A Continuous Time Framework for Discrete Denoising Models*, NeurIPS 2022
* Jo et al., *Score-based Generative Modeling of Graphs via the System of SDEs (GDSS)*, ICML 2022
* Xu et al., *GeoDiff: A Geometric Diffusion Model for Molecular Conformation Generation*, ICLR 2022
* De Bortoli et al., *Riemannian Score-Based Generative Modelling*, NeurIPS 2022
* Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models*, CVPR 2022